In [ ]:
import torch
import torch.nn as nn

#Prototype 1

##Functions

###MFs

In [ ]:
gaussian2_params = ['sigma', 'mu']
def gaussian2(x, p):
    return torch.exp(torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [ ]:
gaussian3_params = ['sigma', 'mu', 'f']
def gaussian3(x, p):
    return torch.exp(-p[:, :, 2] * torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [ ]:
bell_params = ["a", "b", "c"]
def bell(x, p):
    return 1 / (1 + torch.pow(torch.abs((x - p[:, :, 2]) / p[:, :, 0]), 2 * p[:, :, 1]))

In [ ]:
triangular_params = ["a", "b"]
def triangular(x, p):
    return torch.clamp(1 - torch.abs((x - p[:, :, 1]) / p[:, :, 0]), min=0)

In [ ]:
trapezoidal_params = ["a", "b", "c", "d"]
def trapezoidal(x, p):
    return torch.min(torch.clamp((x - p[:, :, 0]) / (p[:, :, 1] - p[:, :, 0]), min=0, max=1), torch.clamp((p[:, :, 3] - x) / (p[:, :, 3] - p[:, :, 2]), min=0, max=1))

###Consequent Functions

In [ ]:
def weighted_linear(x, c, w):
    return (x.matmul(c[:, :-1].transpose(0, 1)) + c[:, -1]).mul(w)

###Other Functions

In [ ]:
def sum(x):
    return torch.sum(x, dim=-1)

###Layers

In [ ]:
#  SE AGREGARÁ LA OTRA INICIALIZACION DE PREMISAS ANTES DE COMENZAR A TESTEAR LOS GRADIENTES Y EL ENTRENAMIENTO
#  SE AGREGARÁ LA OTRA INICIALIZACION DE PREMISAS ANTES DE COMENZAR A TESTEAR LOS GRADIENTES Y EL ENTRENAMIENTO
#  SE AGREGARÁ LA OTRA INICIALIZACION DE PREMISAS ANTES DE COMENZAR A TESTEAR LOS GRADIENTES Y EL ENTRENAMIENTO
class FuzzifyLayer(nn.Module):
    '''
    This class represents de first layer (fuzzify layer) of an ANFIS

    Atributos:
        input_shape : dimensión de los datos de entrada
        features : cantidad de características iniciales
        mf : funcion usada como membership function
        params : lista con los nombres de los parametros
        premises : matriz con los parametros usados en cada nodo de esta capa
    '''
    def __init__(self, input_shape, features=3, mf=gaussian3, params=['sigma', 'mu', 'f'], init_mode=0):
        super(FuzzifyLayer, self).__init__()
        self.input_shape = input_shape
        self.features = features
        self.mf = mf
        self.params = params
        if init_mode == 0:
            self.premises = torch.rand(input_shape[-1], features, len(params))
        elif init_mode == 1:
            #OTRA MANERA PARA INICIALIZAR LA PREMISAS
            self.premises = torch.ones(input_shape[-1], features, len(params))

    def forward(self, x):
        return self.mf(x.unsqueeze(x.dim()), self.premises)

    @property
    def premises_structure(self):
        print("Premises Structure:")
        for i in range(self.input_shape[-1]):
            print(f"    x{i} parameters:")
            for j in range(self.features):
                print(f"        feature {j + 1}:")
                [print(f"            {self.params[k]}: {self.premises[i, j, k]}") for k in range(len(self.params))]


In [ ]:
class FiringLevelsLayer(nn.Module):
    '''
    Clase usada para calcular los firing levels y normalizarlos
    '''
    def __init__(self, ):
        super(FiringLevelsLayer, self).__init__()

    def forward(self, x):
        return torch.nn.functional.normalize(x.prod(dim=x.dim()-2), p=1, dim=-1)

In [ ]:
class ConsequentLayer(nn.Module):
    '''
    Clase que representa la cuarta capa (capa consecuente) de un ANFIS

    Atributos:
        input_shape : dimensión de los datos de entrada
        features : cantidad de características
        function : funcion usada como funcion consequente
        consequents : matriz con los parametros usados en cada nodo de esta capa
    '''
    def __init__(self, input_shape, features=3, function=weighted_linear):
        super(ConsequentLayer, self).__init__()
        self.features = features
        self.input_shape = input_shape
        self.function = function
        #CONSEQUENT PARAMETERS inicializados aleatoriamente de momento
        self.consequents = torch.rand(features, input_shape[-1] + 1)

    def forward(self, x, w):
        return self.function(x, self.consequents, w)

    @property
    def consequents_structure(self):
        print("Consequents Structure:")
        for i in range(self.features):
            print(f"    feature {i + 1} consequent parameters: {self.consequents[i]}")

In [ ]:
class OutputLayer(nn.Module):
    '''
    Clase que representa la quinta capa (capa output) de un ANFIS
    '''
    def __init__(self, function=sum):
        super(OutputLayer, self).__init__()
        self.function = function

    def forward(self, x):
        return self.function(x)

##Strucuture

In [ ]:
class Type3ANFIS(nn.Module):
    def __init__(self, input_shape, initial_features=3):
        super(Type3ANFIS, self).__init__()
        self.layer1 = FuzzifyLayer(input_shape, features=initial_features)
        self.layer23 = FiringLevelsLayer()
        self.layer4 = ConsequentLayer(input_shape, features=initial_features)
        self.layer5 = OutputLayer()

    def forward(self, x):
        output = self.layer1(x)
        output = self.layer4(x, self.layer23(output))
        output = self.layer5(x)
        return output